# Chapter 2 - Lab 6: Retention decision with a sequential workflow (prompt chaining)

A **sequential workflow** (prompt chaining) passes the output of one agent to the next, in a fixed order.

*Finance version*: Retrieval (ratios) -> Analysis (profitability, valuation, risks) -> Decision (**BUY / HOLD / SELL**).
*Customer-care version*: Retrieval (customer snapshot) -> Analysis (customer value, service experience, risks)
-> Decision (**INVEST / MAINTAIN / LIMIT**): how much retention effort the customer warrants.

Each agent returns **structured JSON**, so the next agent (or a CRM system) can consume it reliably.

**Disclaimer**: the data is synthetic; the lab illustrates decision support, not an automated policy.
Customer-treatment decisions must stay fair, explainable and reviewable by a human (see Chapter 11 on fairness testing).

In [ ]:
%pip install -q openai-agents

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

In [ ]:
import json
from agents import Agent, Runner, function_tool

# Mock customer snapshots

In [ ]:
# Mock customer-data platform: a snapshot per customer (current year to date, 2026).
CUSTOMER_SNAPSHOTS = {
    "C1001": {"name": "Priya Sharma", "tier": "Platinum", "tenure_years": 7.6,
              "value": {"lifetime_spend_inr": 485000, "spend_ytd_inr": 74000, "orders_ytd": 6,
                        "margin_contribution_ytd_inr": 13320, "est_annual_margin_inr": 15800},
              "service": {"csat_ytd": 2.9, "csat_prior_year": 4.6, "first_contact_resolution_ytd": 0.40,
                          "avg_resolution_hours_ytd": 96, "open_tickets": 1, "escalations_ytd": 2},
              "risk_signals": {"nps_latest": 3, "complaint_rate_ytd": 0.67, "return_rate_ytd": 0.0,
                               "social_media_mention": True, "legal_threat": True}},
    "C1003": {"name": "Ananya Iyer", "tier": "Silver", "tenure_years": 1.9,
              "value": {"lifetime_spend_inr": 18500, "spend_ytd_inr": 2400, "orders_ytd": 1,
                        "margin_contribution_ytd_inr": 430, "est_annual_margin_inr": 1700},
              "service": {"csat_ytd": 4.8, "csat_prior_year": 4.7, "first_contact_resolution_ytd": 1.0,
                          "avg_resolution_hours_ytd": 6, "open_tickets": 0, "escalations_ytd": 0},
              "risk_signals": {"nps_latest": 9, "complaint_rate_ytd": 0.0, "return_rate_ytd": 0.0,
                               "social_media_mention": False, "legal_threat": False}},
    "C1004": {"name": "Vikram Singh", "tier": "Silver", "tenure_years": 1.4,
              "value": {"lifetime_spend_inr": 96500, "spend_ytd_inr": 55500, "orders_ytd": 4,
                        "margin_contribution_ytd_inr": -2400, "est_annual_margin_inr": -1500},
              "service": {"csat_ytd": 3.5, "csat_prior_year": 3.9, "first_contact_resolution_ytd": 0.71,
                          "avg_resolution_hours_ytd": 41, "open_tickets": 1, "escalations_ytd": 1},
              "risk_signals": {"nps_latest": 5, "complaint_rate_ytd": 0.75, "return_rate_ytd": 0.75,
                               "damaged_in_transit_claims_120d": 3, "social_media_mention": False,
                               "legal_threat": False}},
}

## Tool

In [ ]:
@function_tool
def get_customer_snapshot(customer_id: str) -> dict:
    """Get the value, service and risk snapshot of a NovaMart customer."""
    snap = CUSTOMER_SNAPSHOTS.get(customer_id.strip().upper())
    return snap if snap is not None else {"error": f"Unknown customer {customer_id}"}

MODEL = "gpt-4o-mini"

## Agent 1: Customer Retrieval Agent

In [ ]:
customer_retrieval_agent = Agent(
    name="Customer Retrieval Agent",
    model=MODEL,
    instructions="""
You are the Customer Retrieval Agent.

Tasks:
1) Call the snapshot tool for the requested customer.
2) Output ONLY valid JSON with this schema. Keep the output strictly factual: no interpretation.

{
  "customer_id": "...",
  "snapshot": { ... }      // raw numbers from the tool
}
""",
    tools=[get_customer_snapshot],
)

In [ ]:
retrieval = await Runner.run(customer_retrieval_agent, "Provide the snapshot of customer C1001")
print(retrieval.final_output)

## Agent 2: Customer Analysis Agent

In [ ]:
customer_analysis_agent = Agent(
    name="Customer Analysis Agent",
    model=MODEL,
    instructions="""
You are the Customer Analysis Agent.

Input: JSON from the Retrieval Agent.

Tasks:
1) Interpret the snapshot along 2 dimensions:
   - Customer value (lifetime spend, tenure, margin contribution)
   - Service experience (CSAT trend, first contact resolution, resolution time, escalations)
2) Identify 2-3 key risks (e.g., churn, public complaint, legal, return abuse, unprofitable servicing).
3) Output ONLY valid JSON with this schema:

{
  "customer_id": "...",
  "snapshot": { ... },   // keep from input
  "analysis": {
    "customer_value": { "rating": "high|medium|low", "evidence": [string, ...] },
    "service_experience": { "rating": "good|mixed|poor", "evidence": [string, ...] },
    "risks": [string, ...]
  }
}

Rules:
- Base judgments only on the provided numbers.
- Keep evidence short and specific (quote the numbers).
""",
)

In [ ]:
analysis = await Runner.run(customer_analysis_agent, retrieval.final_output)
print(analysis.final_output)

## Agent 3: Retention Decision Agent

In [ ]:
decision_agent = Agent(
    name="Retention Decision Agent",
    model=MODEL,
    instructions="""
You are the Retention Decision Agent.

Input: JSON from the Customer Analysis Agent.

Tasks: based only on that input:
1) Produce a retention stance:
   - INVEST:   proactive recovery (priority handling, senior owner, goodwill within policy)
   - MAINTAIN: standard service, monitor
   - LIMIT:    policy-only service, no discretionary goodwill (e.g., unprofitable or abusive patterns)
2) Provide 3 bullet reasons grounded in the analysis.
3) Provide a "watchlist": which metrics would change your stance.

Output ONLY valid JSON:

{
  "customer_id": "...",
  "stance": "INVEST|MAINTAIN|LIMIT",
  "summary": {
    "reasons": [string, string, string],
    "watchlist": [string, string, string]
  }
}
""",
)

In [ ]:
decision = await Runner.run(decision_agent, analysis.final_output)
print(decision.final_output)

For Priya (`C1001`) expect **INVEST**: high lifetime value and a collapsing service experience with public
and legal risk. The **watchlist** makes the decision boundary explicit, e.g. "CSAT back above 4.0 after replacement".

# All in one place: run the chain for several customers

In [ ]:
async def retention_pipeline(customer_id: str) -> dict:
    step_1 = await Runner.run(customer_retrieval_agent, f"Provide the snapshot of customer {customer_id}")
    step_2 = await Runner.run(customer_analysis_agent, step_1.final_output)
    step_3 = await Runner.run(decision_agent, step_2.final_output)
    return json.loads(step_3.final_output.strip().removeprefix("```json").removesuffix("```").strip())

for cid in ["C1001", "C1003", "C1004"]:
    result = await retention_pipeline(cid)
    print(f"{cid}: {result['stance']}")
    for r in result["summary"]["reasons"]:
        print("   -", r)

Typical result: C1001 **INVEST**, C1003 **MAINTAIN** (happy, low-value), C1004 **LIMIT** (negative
margin, 75% return rate, three damaged-in-transit claims).

## Discussion
1. Is it fair to give different customers different service levels? Where is the line between segmentation and discrimination?
2. Why do we separate *retrieval* from *analysis* from *decision* instead of one big prompt?
3. Which step would you hand to a human first if a regulator asked "why was this customer treated this way?"